In [0]:
import requests

url = "https://api.exchangerate-api.com/v4/latest/USD"

try:
    response = requests.get(url, timeout=10)
    print("Status code:", response.status_code)
    print("Resposta:", response.json())
except Exception as e:
    print("ERRO ao conectar:", e)

In [0]:
%pip install faker

In [0]:
dbutils.library.restartPython()

In [0]:
import sys
sys.path.append("/Workspace/Users/bruno.quelestech@outlook.com/poc-lakehouse-food-latam/src")

from utils.faker_helper import FakerHelper

fh = FakerHelper(pais="brasil")
print("Nome:", fh.gerar_nome())
print("Cidade:", fh.gerar_cidade())
print("Telefone:", fh.gerar_telefone())

In [0]:
dbutils.library.restartPython()

In [0]:
import sys
sys.path.append("/Workspace/Users/bruno.quelestech@outlook.com/poc-lakehouse-food-latam/src")

from utils.faker_helper import FakerHelper
from utils.scd2_handler import SCD2Handler

print("Importações OK")

In [0]:
from utils.scd2_handler import SCD2Handler

# Criando um DataFrame de exemplo, simulando 2 produtos "crus"
# (sem nenhuma coluna de controle SCD2 ainda)
dados_exemplo = [
    ("PROD001", "MAIONESE", "Maionese", "Mayonesa", "Mayonesa", "Mayonnaise", "1kg"),
    ("PROD002", "MOSTARDA", "Mostarda", "Mostaza", "Mostaza", "Mustard", "5kg"),
]

colunas = [
    "produto_id", "nome_interno", "nome_brasil",
    "nome_argentina", "nome_mexico", "nome_ingles", "tamanho"
]

df_produtos_cru = spark.createDataFrame(dados_exemplo, colunas)

# Aplicando o controle SCD2
scd2 = SCD2Handler()
df_produtos_com_scd2 = scd2.iniciar_controle_scd2(df_produtos_cru)

df_produtos_com_scd2.display()

In [0]:
# Verificação de cobertura de dias na Silver

spark.table("poc_latam_food.silver.fato_vendas") \
    .select("data_venda") \
    .distinct() \
    .orderBy("data_venda") \
    .display()

In [0]:
# Verificação final - Gold também está atualizada

spark.table("poc_latam_food.gold.sales_global").orderBy("period").display()

In [0]:
# Estado atual da Raw - partições e contagens

spark.table("poc_latam_food.raw.vendas") \
    .groupBy("data_ingestao_particao") \
    .count() \
    .orderBy("data_ingestao_particao") \
    .display()

print(f"Total atual na Raw: {spark.table('poc_latam_food.raw.vendas').count()}")

In [0]:
from datetime import datetime, timedelta

data_limite = (datetime.now() - timedelta(hours=48)).date()
print(f"Data-limite (48h atrás): {data_limite}")
print(f"Partições com data_ingestao_particao < '{data_limite}' seriam removidas.")

In [0]:
spark.table("poc_latam_food.raw.vendas") \
    .groupBy("data_ingestao_particao") \
    .count() \
    .orderBy("data_ingestao_particao") \
    .display()

In [0]:
# Contagem por país - partição de hoje

spark.table("poc_latam_food.raw.vendas") \
    .filter("data_ingestao_particao = '2026-07-27'") \
    .groupBy("pais") \
    .count() \
    .orderBy("pais") \
    .display()

In [0]:
# Verificação de duplicidade de venda_id - Argentina, hoje

spark.table("poc_latam_food.raw.vendas") \
    .filter("data_ingestao_particao = '2026-07-27' AND pais = 'argentina'") \
    .groupBy("venda_id") \
    .count() \
    .filter("count > 1") \
    .display()

In [0]:
# Verificação de arquivos na Landing Zone - Argentina, hoje

display(dbutils.fs.ls("/Volumes/poc_latam_food/landing/blob_simulado/vendas/pais=argentina/data=2026-07-27/"))

In [0]:
# Investigação - horários de ingestão das vendas da Argentina, hoje

spark.table("poc_latam_food.raw.vendas") \
    .filter("data_ingestao_particao = '2026-07-27' AND pais = 'argentina'") \
    .groupBy("data_ingestao") \
    .count() \
    .orderBy("data_ingestao") \
    .display()

In [0]:
# Identificação de 500 vendas a remover (arbitrário, mantendo consistência)

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

df_argentina_duplicado = spark.table("poc_latam_food.raw.vendas") \
    .filter("data_ingestao_particao = '2026-07-27' AND pais = 'argentina'")

window_spec = Window.orderBy("venda_id")

df_com_row_number = df_argentina_duplicado.withColumn("rn", row_number().over(window_spec))

ids_para_remover = [row["venda_id"] for row in df_com_row_number.filter("rn > 500").select("venda_id").collect()]

print(f"Total de IDs identificados para remoção: {len(ids_para_remover)}")

In [0]:
# Criando view temporária com os IDs identificados

df_ids_remover = spark.createDataFrame([(vid,) for vid in ids_para_remover], ["venda_id"])
df_ids_remover.createOrReplaceTempView("ids_para_remover")

print(f"View temporária criada com {df_ids_remover.count()} IDs.")

In [0]:
# DELETE na Raw

spark.sql("""
    DELETE FROM poc_latam_food.raw.vendas
    WHERE venda_id IN (SELECT venda_id FROM ids_para_remover)
""")

print(f"Total na Raw após remoção: {spark.table('poc_latam_food.raw.vendas').count()}")

In [0]:
# DELETE na Bronze

spark.sql("""
    DELETE FROM poc_latam_food.bronze.vendas
    WHERE venda_id IN (SELECT venda_id FROM ids_para_remover)
""")

print(f"Total na Bronze após remoção: {spark.table('poc_latam_food.bronze.vendas').count()}")

In [0]:
# Verificação - contagem por país, Bronze, hoje

spark.table("poc_latam_food.bronze.vendas") \
    .filter("data_ingestao_particao = '2026-07-27'") \
    .groupBy("pais") \
    .count() \
    .orderBy("pais") \
    .display()

In [0]:
# Verificação do total atual da Silver (antes de decidir se precisa deletar)

print(f"Total atual na Silver: {spark.table('poc_latam_food.silver.fato_vendas').count()}")

spark.table("poc_latam_food.silver.fato_vendas") \
    .filter("data_venda = '2026-07-27'") \
    .groupBy("pais") \
    .count() \
    .orderBy("pais") \
    .display()

In [0]:
# DELETE na Silver

spark.sql("""
    DELETE FROM poc_latam_food.silver.fato_vendas
    WHERE venda_id IN (SELECT venda_id FROM ids_para_remover)
""")

print(f"Total na Silver após remoção: {spark.table('poc_latam_food.silver.fato_vendas').count()}")

In [0]:
# Validação final - totais em todas as camadas

print(f"Raw: {spark.table('poc_latam_food.raw.vendas').count()}")
print(f"Bronze: {spark.table('poc_latam_food.bronze.vendas').count()}")
print(f"Silver: {spark.table('poc_latam_food.silver.fato_vendas').count()}")

print("\nGold - sales_global:")
spark.table("poc_latam_food.gold.sales_global").orderBy("period").display()

print("\nGold - sales_by_country (Argentina, todos os dias):")
spark.table("poc_latam_food.gold.sales_by_country") \
    .filter("country = 'Argentina'") \
    .orderBy("period") \
    .display()